In [ ]:
import model
import importlib
importlib.reload(model)
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from mesa.visualization import SolaraViz, SpaceRenderer, make_plot_component
from mesa import batch_run
import numpy as np
import sys

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.options.display.float_format = "{:.3f}".format

init_variables = {
    # Variables

    "V": 0,
    "M_A": 0,

    # prior tendencies for neutral behavior
    "theta_N_w0": 3.892,  # should yield around 95% neutral in the beginning
    "theta_N_w1": 0,

    # prior tendencies for friendly behavior
    "theta_F_w0": 0,
    "theta_F_w1": 0,

    # prior tendencies for aggressive behavior
    "theta_A_w0": 0,
    "theta_A_w1": range(-100, 0),

    # Some variables will be calculated in the model or determined randomly.
    # Setting these to anything other than None will raise an exception.
    "theta_N": None,
    "theta_A": None,
    "theta_F": None,
    "p_N": None,
    "p_A": None,
    "p_F": None,
    "r": None,
    "rpe": None,

    # Parameters

    # Parameter for the exponential moving average of anger
    "lambda_A": 0.60,

    # Controllability
    "C": np.linspace(0, 1, 50)
}

results = batch_run(
    model.IrritabilityModel,
    parameters=init_variables,
    data_collection_period=1,  # collect data for every step
    number_processes=16,
    max_steps=100,
)

results_df = pd.DataFrame(results)

results_df.to_parquet("output/001_fnr_impulse_response.parquet",
                      engine="pyarrow")

  0%|          | 0/5000 [00:00<?, ?it/s]

/home/soelderer/projects/formal_model_irritability/simulations/.venv/lib/python3.13/site-packages/mesa/discrete_space/grid.py:103: UserWarning: Random number generator not specified, this can make models non-reproducible. Please pass a random number generator explicitly
  super().__init__(capacity=capacity, random=random, cell_klass=cell_klass)
/home/soelderer/projects/formal_model_irritability/simulations/.venv/lib/python3.13/site-packages/mesa/discrete_space/grid.py:103: UserWarning: Random number generator not specified, this can make models non-reproducible. Please pass a random number generator explicitly
  super().__init__(capacity=capacity, random=random, cell_klass=cell_klass)
/home/soelderer/projects/formal_model_irritability/simulations/.venv/lib/python3.13/site-packages/mesa/discrete_space/grid.py:103: UserWarning: Random number generator not specified, this can make models non-reproducible. Please pass a random number generator explicitly
  super().__init__(capacity=capacit

# Downcast data for dashboard

In [7]:
import pandas as pd
import gc

results_df = pd.read_parquet("output/001_fnr_impulse_response.parquet",
                             engine="pyarrow")

theta_vals = sorted(results_df["theta_A_w1"].unique())
C_vals     = sorted(results_df["C"].unique())

meta_df = pd.DataFrame({
    "theta_vals": [theta_vals],
    "C_vals": [C_vals]
})

# Write to parquet
meta_df.to_parquet("output/dashboard/meta_info.parquet",
                   engine="pyarrow")

results_df.drop(columns=["AgentID", "RunId", "iteration", "r", "rpe", "seed"],
                inplace=True)

for c in results_df.select_dtypes(include="integer"):
    results_df[c] = pd.to_numeric(results_df[c], downcast="integer")

for c in results_df.select_dtypes(include="float"):
    results_df[c] = pd.to_numeric(results_df[c], downcast="float")

results_df.to_parquet(f"output/dashboard/001_fnr_impulse_response.parquet")

# A glimpse at the data

In [ ]:
results_df.head(n=200)

# Prior to the first step, many things are not defined (e.g. action)
results_df = results_df[results_df["Step"] > 0]

results_df.head(n=200)

results_df.columns

# Some plots

In [ ]:
data1 = results_df[(results_df["C"] == 1) &
           (results_df["theta_A_w1"] == -100)]

g = sns.lineplot(data=data1,
                 y="p_A",
                 x="Step",
                 label="Probability aggressive")

g = sns.lineplot(data=data1,
                 y=data1["M_A"].abs(),
                 x="Step",
                 color="Orange",
                 label="Anger/frustration (absolute)")

g.set(title="Anger and aggressive action",
      ylabel="Value");

plt.xlim(0, 40)
plt.ylim(0, 1)
plt.show()